# <center>Laboratorio 9: Benchmark de Carga y Modelos con Spotify 🎵</center>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos</strong></center>

---

### Cuerpo Docente

- Profesores: Pablo Badilla y Diego Cortez
- Auxiliares: Valentina Rojas y Melanie Peña
- Ayudantes: Javiera Arévalo, Tamara Carrasco e Ignacio Reyes

### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Leonardo Navarro
- Nombre de alumno 2: Matias Sweet

---

### Reglas

- **Grupos de 2 personas**
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Prohibido copiar.
- Uso de LLM (Copilot, Claude, Cursor, etc.) restringido a consultas, documentación y corrección de errores.

# Temas a tratar

- Lectura eficiente de datos en formato Parquet.
- Optimización del uso de memoria mediante conversión de tipos de datos.
- Paralelización de operaciones I/O con `ThreadPoolExecutor`.
- Comparación de implementaciones de predicción: Python, NumPy, Numba, pandas y Polars.
- Entrenamiento de modelos con RandomForestRegressor y efecto de `n_jobs`.
- Orquestación de pipelines de datos con Apache Airflow.

# Objetivos principales del laboratorio

- Cargar datos de canciones de Spotify desde archivos Parquet y optimizar su representación en memoria.
- Comparar el tiempo de lectura de archivos en serie vs. en paralelo.
- Analizar el impacto de distintas implementaciones (Python puro, NumPy, Numba, pandas, Polars) en el tiempo de predicción de un modelo lineal.
- Entrenar un RandomForestRegressor que prediga la valencia de canciones, comparando el efecto de la paralelización del entrenamiento.
- Orquestar el pipeline completo (carga + entrenamiento) usando Apache Airflow.

> Instalamos e importamos las librerías necesarias 🎸

In [2]:
!uv pip install pandas pyarrow lightgbm scikit-learn plotly apache-airflow polars numba

Using Python 3.12.9 environment at: C:\MDS7202\.venv
Checked 8 packages in 2.39s


In [3]:
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from pathlib import Path

import numba
import numpy as np
import pandas as pd
import plotly.express as px
import polars as pl
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split

DATA_DIR = Path("data")

# 1. Carga y Optimización de Datos con Parquet

Los datos que usaremos en este laboratorio corresponden a un dataset de canciones de Spotify almacenado en **20 archivos Parquet** (`batch_01.parquet` … `batch_20.parquet`), con un total de 200 000 canciones y 24 columnas que incluyen características de audio, metadatos y la letra completa de cada canción.

A continuación trabajaremos en dos aspectos fundamentales de la carga de datos en la práctica:
1. **Optimizar el uso de memoria** ajustando los tipos de datos de las columnas.
2. **Reducir el tiempo de carga** paralelizando la lectura de archivos.

## 1.1 Exploración y Optimización de Tipos de Datos [1 Punto]

Cuando cargamos datos con pandas, los tipos inferidos por defecto no siempre son los más eficientes. Por ejemplo, un entero que siempre cabe en 16 bits se almacena por defecto como `int64` (64 bits), usando 4 veces más memoria de la necesaria. Lo mismo ocurre con flotantes y con columnas categóricas almacenadas como strings.

**Código dado** — funciones de carga:

In [4]:
def load_batch(path: str) -> pd.DataFrame:
    """Lee un único archivo Parquet y retorna un DataFrame."""
    return pd.read_parquet(path)


def load_all_serial(data_dir: Path, n_batches: int | None = None) -> pd.DataFrame:
    """Lee todos los archivos Parquet de data_dir en serie y los concatena."""
    paths = sorted(data_dir.glob("*.parquet"))
    if n_batches is not None:
        paths = paths[:n_batches]
    return pd.concat([load_batch(str(p)) for p in paths], ignore_index=True)

**TO-DO [0.3 Puntos]:**
- [ ] Ejecutar `load_all_serial` sobre todos los batches y explorar el DataFrame resultante (`.dtypes`, `.memory_usage(deep=True)`).
- [ ] Aplicar las siguientes conversiones a un nuevo DataFrame (copia del originalmente cargado `df_opt`):
  - `float64` → `float32`: columnas de audio features (`danceability`, `energy`, `loudness`, `speechiness`, `acousticness`, `instrumentalness`, `liveness`, `valence`, `tempo`, `avg_artist_popularity`).
  - `int64` → `int16`: columnas `key`, `mode`.
  - `int64` → `int32`: columnas `year`, `popularity`, `duration_ms`, `total_artist_followers`.
- [ ] Comparar el uso de memoria antes y después con un gráfico de barras usando Plotly (código dado).

In [5]:
# Carga de todos los batches en serie
df = load_all_serial(DATA_DIR)

# Exploración de los tipos inferidos y del uso de memoria por columna
display(df.dtypes)
display(df.memory_usage(deep=True))

# Conversiones de tipo para reducir el uso de memoria
dtype_opt = {
    # Audio features: float64 -> float32
    "danceability": "float32",
    "energy": "float32",
    "loudness": "float32",
    "speechiness": "float32",
    "acousticness": "float32",
    "instrumentalness": "float32",
    "liveness": "float32",
    "valence": "float32",
    "tempo": "float32",
    "avg_artist_popularity": "float32",
    # int64 -> int16
    "key": "int16",
    "mode": "int16",
    # int64 -> int32
    "year": "int32",
    "popularity": "int32",
    "duration_ms": "int32",
    "total_artist_followers": "int32",
}

df_opt = df.astype(dtype_opt)

# Compara el uso de memoria antes y después con un gráfico de barras
mem_before = df.memory_usage(deep=True).sum() / 1024**2
mem_after = df_opt.memory_usage(deep=True).sum() / 1024**2

px.bar(
    x=["Antes", "Después"],
    y=[mem_before, mem_after],
    labels={"x": "Estado", "y": "Uso de Memoria (MiB)"},
    title=f"Uso de memoria: {mem_before:.1f} MiB → {mem_after:.1f} MiB ({(1 - mem_after / mem_before) * 100:.1f}% reducción)",
).show()

id                            str
name                          str
album_name                    str
artists                    object
danceability              float64
energy                    float64
key                         int64
loudness                  float64
mode                        int64
speechiness               float64
acousticness              float64
instrumentalness          float64
liveness                  float64
valence                   float64
tempo                     float64
duration_ms                 int64
lyrics                        str
year                        int64
genre                         str
popularity                  int64
total_artist_followers      int64
avg_artist_popularity     float64
artist_ids                 object
niche_genres               object
dtype: object

Index                           132
id                          6000000
name                        5391199
album_name                  5630474
artists                    24000000
danceability                1600000
energy                      1600000
key                         1600000
loudness                    1600000
mode                        1600000
speechiness                 1600000
acousticness                1600000
instrumentalness            1600000
liveness                    1600000
valence                     1600000
tempo                       1600000
duration_ms                 1600000
lyrics                    258813103
year                        1600000
genre                       2639938
popularity                  1600000
total_artist_followers      1600000
avg_artist_popularity       1600000
artist_ids                 24000000
niche_genres               24000000
dtype: int64

### Preguntas [0.7 Puntos]

1. ¿Qué es el formato **Parquet**? ¿Qué ventajas tiene sobre CSV para datos analíticos? ¿Qué es *columnar storage* y por qué acelera las consultas que solo leen algunas columnas?
2. ¿Qué es **Apache Arrow**? ¿Cómo se relaciona con Parquet y con pandas internamente? ¿Qué ganas al usar `pd.read_parquet` en vez de `pd.read_csv`?
3. ¿Por qué existe `float32` si `float64` es más preciso? ¿En qué contextos esa pérdida de precisión es irrelevante?
4. ¿Cuándo **no** conviene reducir la precisión de un tipo numérico? ¿Qué riesgos concretos existen?
5. ¿Existe alguna alternativa a pandas para trabajar con estos datos de forma más eficiente en memoria? (menciona al menos dos)
6. ¿Cuánto se redujo el uso de memoria en total (en MiB y en %)? ¿Era esperable ese resultado? ¿Por qué no se redujo tanto como podría esperarse?
7. ¿Qué pasaría si intentaras reducir `valence` a `float16`? ¿Qué riesgo existiría para el modelo entrenado en la sección 2?

1. Parquet es un formato de almacenamiento en disco orientado a columnas y de código abierto. A diferencia de CSV, que guarda los datos fila por fila como texto, Parquet los agrupa por columna en binario e incluye el esquema dentro del archivo. Eso le da varias ventajas para datos analíticos: comprime mucho mejor, porque los valores de una columna son contiguos y del mismo tipo; no hay que inferir tipos al leer; y permite leer solo algunas columnas. El columnar storage es justamente eso, guardar por columna en vez de por fila, y acelera las consultas que leen pocas columnas porque el motor solo lee del disco esas columnas en lugar de recorrer todas las filas completas.

2. Apache Arrow es un formato estándar de representación columnar en memoria, independiente del lenguaje, pensado para que herramientas como pandas, Polars o Spark compartan datos en memoria sin copiar ni serializar entre formatos. Con Parquet se complementan: Parquet es el formato en disco (almacenar y comprimir) y Arrow el formato en memoria (procesar). pyarrow es quien lee Parquet y lo materializa como tablas Arrow, y pandas la usa como engine por defecto; de hecho en pandas 3.0 los strings ya se respaldan con Arrow. Frente a pd.read_csv, pd.read_parquet es más rápido porque lee binario ya tipado en vez de parsear texto, conserva los dtypes y permite leer solo las columnas necesarias.

3. float32 existe porque ocupa la mitad que float64 (4 bytes contra 8) y suele acelerar los cálculos: caben más datos en caché y se procesan más por instrucción SIMD. float64 da unos 15-16 dígitos significativos y float32 cerca de 7, pero muchas veces esa precisión extra no aporta nada: features acotadas en [0,1], datos cuyo error de medición ya supera el redondeo de float32, o el entrenamiento de modelos de ML, que se hace en float32 sin problema. Cuando el ruido de los datos domina sobre el redondeo, float64 solo gasta memoria.

4. No conviene cuando el rango de los valores supera lo representable por el tipo más chico, o cuando se necesita precisión fina. El riesgo más claro es el overflow: un int16 llega solo a 32767, así que columnas como total_artist_followers o duration_ms se corromperían por wrap-around sin lanzar error (por eso se dejan en int32). También se pierde precisión en sumas o promedios sobre muchos elementos, y nunca conviene guardar IDs como float. Lo peligroso es que estas conversiones fallan en silencio y entregan resultados incorrectos difíciles de detectar.

5. Sí. La más directa es Polars, con DataFrames en Rust respaldados por Arrow, ejecución lazy y multihilo, bastante más eficiente que pandas. Otra es DuckDB, un motor SQL embebido que consulta Parquet directamente sin cargarlo entero a memoria. Para datos que no caben en RAM están Dask o Vaex, que procesan out-of-core.

6. La memoria bajó de 358.7 MiB a 345.7 MiB, apenas un 3.6%. Es esperable, pero la baja es modesta porque las conversiones solo tocan las columnas numéricas, que son una fracción chica del total. El grueso lo consume lyrics, que guarda la letra completa de cada canción y por sí sola pesa cerca de 247 MiB de los ~359 totales, y esa no se modificó. Por eso, aunque las numéricas bajen a la mitad o a un cuarto, el total apenas se mueve. Para una reducción grande habría que actuar sobre las columnas de texto, por ejemplo pasando las de baja cardinalidad a category.

7. valence está en [0,1] y float16 tiene apenas 3-4 dígitos significativos, con resolución cercana a 0.001 en ese rango. Pasarla a float16 metería error de redondeo y valores distintos colapsarían al mismo número. El problema es que valence es el target que el modelo de la sección 2 quiere predecir, así que perder precisión ahí es contaminar las etiquetas: el modelo se entrenaría contra un target redondeado, metiendo ruido en y. Encima float16 casi no está acelerado en CPU, así que scikit-learn lo promovería a float32 igual. No vale la pena arriesgar precisión en el target por ahorrar 2 bytes.


In [6]:
# **IMPORTANTE**: Una vez contestada la pregunta, ejecutar esta celda para liberar memoria.
df_opt = None

## 1.2 Lectura en Serie vs. Paralelo [1 Punto]

Cuando se trabaja con múltiples archivos, la lectura **en paralelo** puede reducir el tiempo total al aprovechar que la espera de I/O (disco/red) no bloquea al procesador. En Python, la clase `ThreadPoolExecutor` del módulo `concurrent.futures` permite lanzar múltiples hilos para ejecutar operaciones de forma concurrente.

**TO-DO: [0.3 Puntos]**
- [ ] Implementar `load_all_parallel` usando `ThreadPoolExecutor`.
- [ ] Medir con `%timeit` ambas versiones sobre todos los batches.
- [ ] Generar un gráfico de línea (Plotly) con los tiempos para 2, 4, 6, …, 20 archivos, con series `Serial` y `Paralelo`.

In [7]:
# Escribe aquí tu código
def load_all_parallel(data_dir: Path, n_batches: int | None = None) -> pd.DataFrame:
    """Lee todos los archivos Parquet de data_dir en paralelo (threads) y los concatena."""
    paths = sorted(data_dir.glob("*.parquet"))
    if n_batches is not None:
        paths = paths[:n_batches]
    with ThreadPoolExecutor() as executor:
        dfs = list(executor.map(load_batch, [str(p) for p in paths]))
    return pd.concat(dfs, ignore_index=True)

In [8]:
# Medición con %timeit sobre los 20 batches
print("Serial:")
%timeit -n 1 -r 5 load_all_serial(DATA_DIR)
print("Paralelo:")
%timeit -n 1 -r 5 load_all_parallel(DATA_DIR)

Serial:
896 ms ± 54.4 ms per loop (mean ± std. dev. of 5 runs, 1 loop each)
Paralelo:
437 ms ± 9.33 ms per loop (mean ± std. dev. of 5 runs, 1 loop each)


**Benchmark:** mide tiempos para 2, 4, 6, ..., 20 archivos y grafica


In [9]:
@dataclass
class ReadMeasurement:
    n_files: int
    time_sec: float
    version: str


measurements: list[ReadMeasurement] = []

for n in range(2, 21):
    t0 = time.perf_counter()
    load_all_serial(DATA_DIR, n_batches=n)
    measurements.append(ReadMeasurement(n, time.perf_counter() - t0, "Serial"))

    t0 = time.perf_counter()
    load_all_parallel(DATA_DIR, n_batches=n)
    measurements.append(ReadMeasurement(n, time.perf_counter() - t0, "Paralelo"))

df_times = pd.DataFrame(measurements)
px.line(
    df_times,
    x="n_files",
    y="time_sec",
    color="version",
    markers=True,
    title="Tiempo de lectura: Serial vs Paralelo",
    labels={"n_files": "Número de archivos", "time_sec": "Tiempo (s)"},
).show()

### Preguntas  [0.7 Puntos]

1. ¿Qué significa que una operación sea **I/O-bound** vs **CPU-bound**? ¿A cuál categoría pertenece la lectura de archivos desde disco?
2. ¿Qué es el **GIL** (*Global Interpreter Lock*) de CPython? ¿Por qué existe? ¿Qué problema resuelve y qué limitación introduce?
3. ¿Por qué usamos Python si tiene el GIL? ¿Qué ganamos al usarlo como lenguaje de *pegamento* entre librerías de alto rendimiento (NumPy, Arrow, PyTorch…)?
4. ¿Cuándo conviene usar `ThreadPoolExecutor` vs `ProcessPoolExecutor`? ¿Cuál usarías si la operación fuera puramente CPU-bound?
5. ¿Qué overhead introduce crear un pool de threads? ¿Qué pasaría si los archivos fueran muy pequeños (p.ej. 1 KB cada uno)?
6. ¿Se observó mejora con la lectura paralela? ¿A partir de cuántos archivos empieza a ser notable?
7. ¿Por qué el speedup obtenido **no es igual** al número de threads disponibles? ¿Qué factores lo limitan?

1. Una operación es I/O-bound cuando su cuello de botella es la espera de entrada/salida (disco, red, base de datos): el procesador pasa el tiempo esperando los datos más que calculando. Es CPU-bound cuando el cuello de botella es el cómputo y el procesador está al máximo sin esperar nada externo. Leer archivos desde disco es I/O-bound, porque el tiempo se va sobre todo en esperar que el disco entregue los bytes; el parseo posterior tiene algo de CPU, pero la espera domina.

2. El GIL (Global Interpreter Lock) es un mutex de CPython que deja ejecutar bytecode de Python a un solo hilo a la vez dentro de un proceso. Existe porque la gestión de memoria de CPython usa conteo de referencias, que no es thread-safe, y el GIL evita condiciones de carrera sobre esos contadores sin tener que poner locks en cada objeto. Su limitación es que dos hilos no pueden correr código Python en paralelo, así que el multithreading no acelera tareas CPU-bound puras.

3. A pesar del GIL, Python es productivo y legible, y funciona muy bien como lenguaje de pegamento: el trabajo pesado lo hacen librerías en C, C++ o Rust (NumPy, Arrow, PyTorch) que liberan el GIL mientras ejecutan sus rutinas de bajo nivel. Así uno escribe la lógica en Python pero el cómputo intensivo ocurre fuera del intérprete, en código compilado y muchas veces vectorizado o paralelizado. Como además sueltan el GIL durante I/O o cálculo, el multithreading sí ayuda en esos casos.

4. ThreadPoolExecutor conviene para tareas I/O-bound: mientras un hilo espera I/O suelta el GIL y otro avanza, y los hilos son baratos y comparten memoria. ProcessPoolExecutor conviene para CPU-bound, porque cada proceso tiene su propio intérprete y su propio GIL, así que el cálculo corre en paralelo en varios cores; el costo es que crear procesos es más caro y hay que serializar los datos entre ellos. Si la operación fuera puramente CPU-bound usaría ProcessPoolExecutor.

5. Crear un pool implica levantar y administrar los hilos, repartir las tareas en una cola y recoger los resultados, lo que tiene un costo fijo. Si los archivos fueran muy pequeños (1 KB), ese overhead dominaría: cada lectura tardaría tan poco que coordinar los hilos costaría tanto o más que leer, y la versión paralela podría terminar más lenta que la serial. El paralelismo rinde cuando cada tarea es lo bastante grande como para justificar el costo de repartirla.

6. Sí, la curva paralela queda por debajo de la serial en todo el rango. Incluso con 2 archivos ya hay ganancia (cerca de 1.6×), aunque la diferencia absoluta es chica. Al aumentar los archivos el speedup se mantiene más o menos estable, entre 1.5× y 2×, y la brecha en tiempo se hace más visible desde unos 8-10 archivos, donde la serial ronda el medio segundo y sigue subiendo mientras la paralela se queda más abajo. Sobre los 20 batches el %timeit lo confirma: 896 ms en serial contra 437 ms en paralelo, alrededor de 2×.

7. El speedup no iguala al número de threads porque la lectura no es perfectamente paralelizable. Hay una parte serial inevitable (crear el pool, repartir tareas y el pd.concat final), el disco es un recurso compartido de ancho de banda limitado por el que los hilos compiten, y el GIL solo se libera durante el I/O, no durante el parseo en Python. Por la ley de Amdahl esa fracción serial pone un techo, así que aunque haya muchos hilos la mejora real (aquí cercana a 2×) queda por debajo.


# 2. Predicción de Valencia

La columna `valence` de Spotify mide el **positivismo musical** de una canción: valores cercanos a 1 indican canciones alegres y eufóricas, mientras que valores cercanos a 0 corresponden a canciones tristes o melancólicas. En esta sección analizaremos distintas formas de realizar predicciones con un modelo de regresión lineal ya entrenado, y luego entrenaremos un modelo más complejo.

## 2.1 Regresión Lineal a Mano [1.5 Puntos]

Antes de entrenar un modelo completo, veremos cómo **la elección de implementación** afecta drásticamente el rendimiento de predicción. Usaremos un modelo de regresión lineal pre-entrenado cuyos coeficientes ya están dados, e implementaremos la predicción usando cinco enfoques distintos: Python puro, NumPy, Numba (JIT), pandas y Polars.

**Código dado — carga de datos y parámetros del modelo:**

In [10]:
# Carga de datos y preparación del split
df_train = load_all_serial(DATA_DIR, n_batches=20)

PARAM_COLS = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "tempo",
    "duration_ms",
    "year",
]

X = df_train[PARAM_COLS + ["key", "mode", "genre"]]
y = df_train["valence"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
# Parámetros del modelo lineal pre-entrenado (dados)
params = {
    "danceability": 0.7718203106411208,
    "energy": 0.4252134896942928,
    "loudness": -0.008319917439312445,
    "speechiness": -0.24543273088867107,
    "acousticness": 0.10440236785191129,
    "instrumentalness": -0.11203723673701874,
    "liveness": 0.023790522969698424,
    "tempo": 0.0007885690378158087,
    "duration_ms": -4.31739613602265e-07,
    "year": -0.0036043842721972985,
}
intercept = 6.948154825159983

params_vals = list(params.values())
params_arr = np.array(params_vals, dtype=np.float32)

**Código dado — las 5 implementaciones de predicción:**

Analiza cómo cada implementación aborda el mismo problema y presta atención a las diferencias en legibilidad, concisión y (como verás en el benchmark) rendimiento.

In [12]:
def linear_regression_predict(X: np.ndarray, params: list[float]) -> list[float]:
    """Predicción con loop Python puro."""
    preds = []
    for row in X:
        val = intercept
        for j, w in enumerate(params):
            val += row[j] * w
        preds.append(val)
    return preds


def linear_regression_predict_numpy(X: np.ndarray, params: list[float]) -> np.ndarray:
    """Predicción vectorizada con NumPy."""
    return (X * np.array(params)).sum(axis=1) + intercept


@numba.njit
def linear_regression_predict_numba(X: np.ndarray, params: np.ndarray) -> np.ndarray:
    """Predicción con Numba JIT (loop compilado a código máquina)."""
    n = X.shape[0]
    preds = np.empty(n)
    for i in range(n):
        val = intercept
        for j in range(len(params)):
            val += X[i, j] * params[j]
        preds[i] = val
    return preds


def linear_regression_predict_pandas(X: pd.DataFrame, params: list[float]) -> pd.Series:
    """Predicción vectorizada con pandas (dot product)."""
    return X.dot(pd.Series(params, index=X.columns)) + intercept


def linear_regression_predict_polars(X: pl.DataFrame, params: list[float]) -> pl.Series:
    """Predicción vectorizada con Polars (expresiones lazy)."""
    weights = dict(zip(X.columns, params, strict=False))
    expr = pl.lit(intercept)
    for col, w in weights.items():
        expr = expr + pl.col(col) * w
    return X.select(expr.alias("pred"))["pred"]

**Código dado — Benchmark de las 5 implementaciones:**

In [13]:
@dataclass
class TimeMeasurement:
    time_took: float
    iteration: int
    version: str


time_measurements: list[TimeMeasurement] = []

ranges = [10, 50, 100, 250, 500, 750, 1000, *range(1001, len(X_test) + 1, 1000)]

for it in ranges:
    X_np = X_test[PARAM_COLS].iloc[:it].to_numpy(dtype=np.float32)
    X_pd = X_test[PARAM_COLS].iloc[:it]
    X_pl = pl.from_pandas(X_pd)

    for name, fn, args in [
        ("Python", linear_regression_predict, (X_np, params_vals)),
        ("NumPy", linear_regression_predict_numpy, (X_np, params_vals)),
        ("Numba-JIT", linear_regression_predict_numba, (X_np, params_arr)),
        ("Pandas", linear_regression_predict_pandas, (X_pd, params_vals)),
        ("Polars", linear_regression_predict_polars, (X_pl, params_vals)),
    ]:
        t0 = time.perf_counter()
        fn(*args)
        time_measurements.append(TimeMeasurement(time.perf_counter() - t0, it, name))

df_bench = pd.DataFrame(time_measurements)

# Gráfico 1: tiempos absolutos
px.line(
    df_bench,
    x="iteration",
    y="time_took",
    color="version",
    markers=True,
    title="Tiempos de predicción según implementación",
    labels={"iteration": "Número de filas", "time_took": "Tiempo (s)"},
).show()

# Gráfico 2: tiempos absolutos (en log)
px.line(
    df_bench,
    x="iteration",
    y="time_took",
    color="version",
    markers=True,
    title="Tiempos de predicción según implementación (en escala logarítmica)",
    labels={"iteration": "Número de filas", "time_took": "Tiempo (s)"},
    log_y=True,
).show()

# Gráfico 3: speedup relativo respecto a Python puro
pivot = df_bench.pivot(index="iteration", columns="version", values="time_took")
for col in ["NumPy", "Numba-JIT", "Pandas", "Polars"]:
    pivot[col] = pivot["Python"] / pivot[col]
pivot["Python"] = 1.0

melted = pivot.reset_index().melt(
    id_vars=["iteration"],
    value_vars=["Python", "NumPy", "Numba-JIT", "Pandas", "Polars"],
    value_name="speedup",
)
px.line(
    melted,
    x="iteration",
    y="speedup",
    color="version",
    markers=True,
    title="Speedup relativo respecto a Python puro",
    labels={"iteration": "Número de filas", "speedup": "Speedup (×)"},
).show()

In [14]:
# Versión de pandas instalada
pd.__version__

'3.0.1'

### Preguntas [1.5 Puntos]

  1. ¿Qué es la vectorización en NumPy? ¿Cómo puede ejecutar operaciones sobre arrays sin loops de Python explícitos?
  2. ¿Qué es JIT (Just-In-Time compilation)? ¿Qué hace el decorador @numba.njit? ¿Qué significa el modo nopython?
  3. ¿Por qué Numba es más lento en la primera ejecución? ¿Qué es el warm-up de JIT y cómo lo manejamos en el benchmark?
  4. ¿Qué es Polars y cuáles son sus principales características como librería de datos? ¿Para qué escenarios fue diseñada y por qué ha ganado popularidad como alternativa a pandas?
  5. ¿En qué se diferencia Polars de pandas a nivel de implementación (lenguaje, modelo de ejecución, manejo de memoria)?
  6. ¿Por qué pandas puede ser más lento que NumPy aun usando operaciones vectorizadas internamente?
  7. ¿Qué son las instrucciones SIMD (Single Instruction Multiple Data)? ¿Cómo contribuyen a la aceleración de NumPy y Polars?
  8. ¿Cuándo conviene usar Numba sobre NumPy? ¿Y Polars sobre pandas para operaciones numéricas?
  9. ¿Cuál implementación fue la más rápida en tu medición? ¿Era esperable ese resultado?
  10. ¿Se observa diferencia notable entre pandas y NumPy? ¿Por qué pandas puede ser más lento o más rápido?
  11. ¿A partir de cuántas filas empieza a ser evidente la ventaja de NumPy/Numba sobre Python puro?
  12. ¿Polars fue más eficiente que pandas en tu medición? Verifica la versión de pandas instalada (pd.__version__) y comenta si crees que la versión influye en el resultado.
  13. ¿Por qué Numba puede igualar o superar a NumPy para loops numéricos simples?
  14. El benchmark excluye el costo de convertir datos a NumPy/Polars (la conversión ocurre fuera del timing). ¿Cómo cambiaría el resultado si incluyeras ese costo? ¿En qué escenarios de
  producción ese costo no existiría?
  15.  Si tuvieras que realizar esta predicción sobre 100 millones de filas en un servidor de producción, ¿qué implementación elegirías y por qué? ¿Cambiaría tu respuesta si dispusieras de
  una GPU?

1. La vectorización consiste en expresar una operación sobre todo el array de una vez, en lugar de iterar elemento por elemento en Python. NumPy puede hacerlo sin loops explícitos porque sus arrays son bloques contiguos de memoria de un solo tipo, y las operaciones se delegan a rutinas precompiladas en C que recorren ese bloque internamente. Así el loop ocurre en código nativo, sin el overhead del intérprete, y muchas veces aprovechando instrucciones SIMD que procesan varios elementos por ciclo.

2. JIT (Just-In-Time) es compilar el código a código máquina en tiempo de ejecución, justo antes de usarlo, en vez de interpretarlo. El decorador @numba.njit toma la función Python y, la primera vez que se llama, la compila a código nativo especializado para los tipos de los argumentos. El modo nopython significa que Numba compila sin recurrir al intérprete de Python ni a objetos Python intermedios; si lo logra el resultado es muy rápido, y si no puede lanza error en vez de degradar el rendimiento (njit es justamente jit con nopython=True).

3. Porque la primera llamada incluye el costo de compilar la función a código máquina, que es caro. En el benchmark se ve clarísimo: la primera medición de Numba fue de 1.09 s, contra microsegundos en las siguientes. Ese costo único de compilación es el warm-up del JIT. Como el benchmark mide sobre tamaños crecientes, la compilación se paga una sola vez en la primera llamada y el resto de la curva ya usa la versión compilada, así que refleja el rendimiento real sin el warm-up.

4. Polars es una librería de DataFrames escrita en Rust y respaldada por Apache Arrow. Sus rasgos principales son la ejecución multihilo por defecto, un modo lazy con optimización de consultas y un manejo de memoria muy eficiente gracias a Arrow. Fue diseñada para procesar datos tabulares grandes de forma rápida y con bajo consumo de memoria, aprovechando todos los cores. Ganó popularidad como alternativa a pandas porque suele ser bastante más rápida, escala mejor y tiene una API expresiva, sin dejar de correr en una sola máquina.

5. Pandas está implementado sobre NumPy y Python, su ejecución es mayormente eager y single-thread, y maneja la memoria con bloques de NumPy. Polars está escrito en Rust, ofrece ejecución lazy con un optimizador de consultas además del modo eager, es multihilo de forma nativa y guarda los datos en formato Arrow columnar. En la práctica eso le permite paralelizar operaciones y evitar copias, mientras que pandas históricamente corre en un solo hilo y arrastra más overhead por su modelo basado en NumPy.

6. Porque pandas trabaja por encima de NumPy y agrega overhead propio: alinear por índice, validar dtypes, considerar valores faltantes y construir objetos Series/DataFrame con sus etiquetas. Para una operación tan simple como el producto punto, ese trabajo alrededor del cálculo pesa más que el cálculo en sí, sobre todo con pocos datos. Por eso en el benchmark pandas quedó algo por detrás de NumPy (0.0018 contra 0.0014 s en promedio sobre los tamaños grandes).

7. SIMD (Single Instruction, Multiple Data) son instrucciones del procesador que aplican una misma operación a varios datos a la vez, usando registros que contienen varios valores (por ejemplo 8 floats). Aceleran a NumPy y Polars porque sus datos están en bloques contiguos y homogéneos, lo que permite que el código nativo procese varios elementos por instrucción en vez de uno por uno. Eso multiplica el throughput en operaciones aritméticas como sumas o productos sobre arrays.

8. Numba conviene cuando la lógica no se expresa bien de forma vectorizada: loops con dependencias, condiciones complejas o cálculos elemento a elemento que en NumPy obligarían a crear muchos arrays intermedios; ahí compila el loop a código nativo y suele ganar. Polars conviene sobre pandas cuando los datos son grandes y la operación se beneficia del paralelismo y de la ejecución lazy (filtros, agregaciones y joins sobre muchas filas), donde su motor multihilo y Arrow marcan diferencia.

9. La más rápida fue Numba-JIT una vez compilada, con cerca de 0.00034 s sobre las ~38 mil filas, más de 250× más rápida que Python puro. Sí era esperable: Numba compila el loop a código máquina especializado, así que evita tanto el overhead del intérprete como la creación de arrays intermedios que sí tiene NumPy. Eso sí, paga un warm-up grande en la primera llamada (1.09 s), que es el precio de compilar.

10. La diferencia es chica: en los tamaños grandes NumPy promedió 0.0014 s y pandas 0.0018 s, así que NumPy quedó levemente por delante. pandas es algo más lento porque envuelve a NumPy y agrega overhead (índices, dtypes, construcción de Series). En tamaños muy pequeños esa diferencia relativa es mayor, porque pandas arrastra un costo fijo de varios cientos de microsegundos casi independiente del número de filas, mientras NumPy parte mucho más liviano.

11. Con muy pocas filas (10-100) Python puro es competitivo e incluso más rápido, porque el costo de armar el array o llamar a la rutina nativa no se amortiza, y Numba ni siquiera ha compilado. La ventaja se vuelve evidente alrededor de unos cientos de filas: hacia las 500-1000 filas Python ya es claramente más lento (del orden de 0.001-0.002 s contra ~0.0001 s de NumPy), y la brecha solo crece desde ahí, hasta los dos órdenes de magnitud en los tamaños grandes.

12. Sí, Polars fue más rápido que pandas en los tamaños grandes (0.0009 contra 0.0018 s, cerca de 2×), gracias a su motor en Rust, multihilo y respaldado por Arrow. La versión instalada es pandas 3.0.1, que es reciente e incorpora mejoras como Copy-on-Write y strings sobre Arrow. Aun así, para una operación tan simple como este producto punto sobre 10 features esas mejoras aportan poco, así que la diferencia se debe sobre todo al diseño de Polars más que a la versión de pandas; con una versión más antigua el resultado sería parecido o algo peor para pandas.

13. Porque NumPy ejecuta cada operación vectorizada como un paso separado y suele crear arrays temporales intermedios (por ejemplo X*params y luego el sum), lo que implica recorrer la memoria varias veces y reservar buffers. Numba compila todo el loop en una sola pasada, fusionando las operaciones y sin materializar intermedios, leyendo cada dato una vez. Para loops simples eso le permite igualar o superar a NumPy, que paga el costo de los temporales y de varias pasadas por los datos.

14. Si se incluyera la conversión (to_numpy, pl.from_pandas), cada implementación cargaría con el costo de mover y copiar los datos a su formato antes de calcular, y eso puede ser mayor que la predicción misma cuando el cálculo es tan barato; las que más sufrirían son Polars y Numba, que requieren pasar los datos a su representación. En producción ese costo no existe cuando los datos ya viven en el formato adecuado: si la fuente ya entrega Arrow/Polars, o si el pipeline trabaja directamente con arrays NumPy, no hay conversión que pagar.

15. Para 100 millones de filas descartaría Python puro de inmediato. Entre las demás elegiría según el contexto: si el cálculo ya está vectorizado y los datos vienen en Arrow/Polars, Polars escala muy bien por ser multihilo y eficiente en memoria; si la lógica es un loop numérico personalizado, Numba compilado sería muy competitivo. Con una GPU la respuesta cambiaría: para ese volumen convendría una librería sobre GPU (cuDF, CuPy o un framework como PyTorch/JAX), que paraleliza el producto punto sobre miles de núcleos y supera ampliamente a cualquier opción de CPU, siempre que los datos quepan en su memoria y el costo de transferirlos se amortice.


### 2.2 Entrenamiento y Comparación de `n_jobs` [0.5 Puntos]

Ahora entrenaremos un modelo más complejo: un **RandomForestRegressor** que usa las características de audio más una codificación del género musical para predecir `valence`. Compararemos el efecto de paralelizar el entrenamiento con el parámetro `n_jobs`.

**Código dado — pipeline encapsulado** (no modificar):

In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


def build_pipeline(n_jobs: int = 1) -> Pipeline:
    # En producción este pipeline usaría LGBMRegressor; aquí usamos RandomForest
    # para ilustrar el efecto de n_jobs de forma más pronunciada.
    return Pipeline(
        [
            (
                "column_transformer",
                ColumnTransformer(
                    [
                        ("ohe", OneHotEncoder(handle_unknown="ignore"), ["key", "mode", "genre"]),
                        (
                            "numerical",
                            "passthrough",
                            PARAM_COLS,
                        ),
                    ]
                ),
            ),
            ("random_forest", RandomForestRegressor(n_jobs=n_jobs, random_state=42)),
        ]
    )

In [16]:
# Entrena con n_jobs=1 y mide el tiempo
pipeline_1 = build_pipeline(n_jobs=1)
t0 = time.perf_counter()
pipeline_1.fit(X_train, y_train)
time_1job = time.perf_counter() - t0

# Entrena con n_jobs=-1 y mide el tiempo
pipeline_all = build_pipeline(n_jobs=-1)
t0 = time.perf_counter()
pipeline_all.fit(X_train, y_train)
time_all_jobs = time.perf_counter() - t0

# Calcula RMSE de ambos modelos
rmse_1 = root_mean_squared_error(y_test, pipeline_1.predict(X_test))
rmse_all = root_mean_squared_error(y_test, pipeline_all.predict(X_test))

print(f"n_jobs=1  → tiempo: {time_1job:.1f}s | RMSE: {rmse_1:.4f}")
print(f"n_jobs=-1 → tiempo: {time_all_jobs:.1f}s | RMSE: {rmse_all:.4f}")

n_jobs=1  → tiempo: 198.0s | RMSE: 0.1652
n_jobs=-1 → tiempo: 42.1s | RMSE: 0.1652


In [17]:
# Gráficos de tiempos y RMSE
df_perf = pd.DataFrame(
    {
        "configuracion": ["n_jobs=1", "n_jobs=-1"],
        "tiempo_s": [time_1job, time_all_jobs],
        "rmse": [rmse_1, rmse_all],
    }
)

px.bar(
    df_perf,
    x="configuracion",
    y="tiempo_s",
    title="Tiempo de entrenamiento según n_jobs",
    labels={"tiempo_s": "Tiempo (s)", "configuracion": "Configuración"},
    text_auto=".1f",
).show()

px.bar(
    df_perf,
    x="configuracion",
    y="rmse",
    title="RMSE según n_jobs",
    labels={"rmse": "RMSE", "configuracion": "Configuración"},
    text_auto=".4f",
).show()

### Preguntas [0.5 Puntos]

1. ¿Qué hace el parámetro `n_jobs` en RandomForest (y en general en scikit-learn)?
2. **¿Por qué aquí sí funciona el paralelismo real sin el problema del GIL?** (Pista: RandomForest en scikit-learn usa joblib con backend de procesos o threads nativos.)
3. ¿Cuánto mejoró el tiempo con `n_jobs=-1`? 
4. ¿Fue proporcional al número de CPUs disponibles en tu máquina? ¿Por qué no?
5. ¿Hubo diferencia en RMSE entre ambas versiones? ¿Era esperable? ¿Por qué?

1. n_jobs indica cuántos trabajos en paralelo usar para las partes paralelizables del algoritmo. En RandomForest controla cuántos árboles se entrenan al mismo tiempo, repartiéndolos entre varios núcleos, lo que es natural porque en bagging cada árbol es independiente. En scikit-learn en general aparece en operaciones que se pueden dividir en tareas independientes (cross-validation, GridSearch, vecinos, etc.). n_jobs=1 usa un solo core y n_jobs=-1 usa todos los disponibles.

2. Porque RandomForest paraleliza con joblib, que para tareas CPU-bound usa por defecto un backend de procesos (loky). Cada proceso tiene su propio intérprete y su propio GIL, así que los árboles se entrenan de verdad en paralelo en varios cores sin que el GIL los serialice. Además, buena parte del cálculo de los árboles ocurre en código compilado (Cython) que libera el GIL. Por eso, a diferencia del multithreading en Python puro, aquí el paralelismo sí se traduce en aceleración real.

3. Bastante: el entrenamiento pasó de 198.0 s con n_jobs=1 a 42.1 s con n_jobs=-1, un speedup cercano a 4.7×.

4. No del todo. La máquina tiene 8 procesadores lógicos y el speedup fue de ~4.7×, por debajo de 8. Hay varias razones: parte del trabajo no se paraleliza (el OneHotEncoder del ColumnTransformer, la preparación de datos y la agregación final), lo que por la ley de Amdahl pone un techo; lanzar y coordinar los procesos tiene overhead y los datos deben copiarse a cada worker; los cores compiten por memoria y caché; y esos 8 son procesadores lógicos (hyperthreading), que no rinden como 8 núcleos físicos. Todo eso deja la mejora real por debajo del ideal.

5. No, el RMSE fue idéntico en ambos casos (0.1652). Era completamente esperable: n_jobs solo cambia cómo se reparte el trabajo entre cores, no qué se calcula. Con random_state=42 fijo, el bosque entrenado es exactamente el mismo se use 1 o todos los cores, así que produce las mismas predicciones y el mismo error. El paralelismo afecta el tiempo, no el resultado.


# 3. Orquestación del Pipeline con Apache Airflow

En producción, los pipelines de datos y ML rara vez se ejecutan a mano desde un notebook. Se necesita:
- **Automatización**: que el pipeline corra periódicamente (diariamente, por hora…).
- **Dependencias**: que el entrenamiento solo comience si la carga de datos terminó exitosamente.
- **Monitoreo y reintentos**: que si una tarea falla, el sistema lo registre y reintente.

**Apache Airflow** resuelve exactamente esto. Define pipelines como **DAGs** (*Directed Acyclic Graphs*), donde cada nodo es una **tarea** y las aristas definen dependencias.

| Concepto | Descripción |
|----------|-------------|
| **DAG** | Grafo Dirigido Acíclico que representa el pipeline completo |
| **Operator** | Unidad de trabajo (`PythonOperator`, `BashOperator`, …) |
| **Task** | Instancia de un Operator dentro de un DAG |
| **XCom** | Mecanismo para pasar datos pequeños entre tareas |
| **schedule** | Expresión cron que indica cuándo ejecutar el DAG |

### Setup local


En la carpeta del Lab:

```bash
export AIRFLOW_HOME=$(pwd)
airflow db migrate          # inicializa la base de datos de metadata
# Ver la contraseña. Si no se en un comienzo, ejecutar airflow standalone, parar el proceso y luego ejecutar nuevamente este comando. 
cat $AIRFLOW_HOME/simple_auth_manager_passwords.json.generated 
airflow standalone       # levanta scheduler + webserver en http://localhost:8080
```

Los DAGs deben guardarse en `./dags`.

## 3.1 Implementación del DAG

**TO-DO [0.8 Puntos]:**
- [ ] Implementar `task_load_data_fn`: cargar 5 batches en paralelo, guardar en disco como Parquet y pasar la ruta a la siguiente tarea usando XCom.
- [ ] Implementar `task_train_model_fn`: recuperar la ruta de XCom, cargar el DataFrame, preparar X e y, entrenar `build_pipeline(n_jobs=-1)` e imprimir el tiempo.
- [ ] Definir la dependencia entre tareas (`load_data >> train_model`).

El siguiente bloque es el template que debes completar en tu celda de respuesta.

In [ ]:
%%writefile ~/airflow/dags/spotify_pipeline_dag.py

from pathlib import Path
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split


DATA_DIR = Path("/RUTA/ABSOLUTA/A/Labs/Lab9_v2/data")  # AJUSTA esta ruta
OUTPUT_PATH = Path("/tmp/spotify_data.parquet")

PARAM_COLS = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "tempo",
    "duration_ms",
    "year",
]


# ── Funciones auxiliares (dadas) ─────────────────────────────────────────────


def load_batch(path: str) -> pd.DataFrame:
    return pd.read_parquet(path)


def load_all_parallel(data_dir: Path, n_batches: int = 5) -> pd.DataFrame:
    paths = sorted(data_dir.glob("*.parquet"))[:n_batches]
    with ThreadPoolExecutor(max_workers=None) as executor:
        dfs = list(executor.map(load_batch, [str(p) for p in paths]))
    return pd.concat(dfs, ignore_index=True)


def build_pipeline(n_jobs: int = -1) -> Pipeline:
    return Pipeline(
        [
            (
                "column_transformer",
                ColumnTransformer(
                    [
                        ("ohe", OneHotEncoder(handle_unknown="ignore"), ["key", "mode", "genre"]),
                        ("numerical", "passthrough", PARAM_COLS),
                    ]
                ),
            ),
            ("random_forest", RandomForestRegressor(n_jobs=n_jobs, random_state=42)),
        ]
    )


# ── Funciones de las tareas de Airflow ───────────────────────────────────────


def task_load_data_fn(**context):
    """
    Carga 5 batches de datos en paralelo y guarda el resultado en disco.
    TODO: implementa esta función.
    - Usa load_all_parallel para cargar los datos.
    - Guarda el DataFrame resultante en OUTPUT_PATH (formato parquet).
    - Usa XCom para pasar la ruta del archivo a la siguiente tarea.
    """
    ...


def task_train_model_fn(**context):
    """
    Carga los datos desde disco y entrena el pipeline.
    TODO: implementa esta función.
    - Recupera la ruta del archivo desde XCom.
    - Lee el DataFrame desde esa ruta.
    - Prepara X e y, realiza el split 80/20.
    - Entrena build_pipeline(n_jobs=-1).
    - Imprime el tiempo de entrenamiento.
    """
    ...


# ── Definición del DAG ────────────────────────────────────────────────────────

with DAG(
    dag_id="spotify_pipeline",
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
    tags=["mds7202", "spotify"],
) as dag:
    load_data = PythonOperator(
        task_id="load_data",
        python_callable=task_load_data_fn,
    )

    train_model = PythonOperator(
        task_id="train_model",
        python_callable=task_train_model_fn,
    )

    # TODO: define la dependencia entre tareas (load_data debe ejecutarse antes que train_model)
    ...


In [19]:
%%writefile dags/spotify_pipeline_dag.py

from pathlib import Path
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split


DATA_DIR = Path("/mnt/c/MDS7202/labs/lab-9/data")
OUTPUT_PATH = Path("/tmp/spotify_data.parquet")

PARAM_COLS = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "tempo",
    "duration_ms",
    "year",
]


# ── Funciones auxiliares (dadas) ─────────────────────────────────────────────


def load_batch(path: str) -> pd.DataFrame:
    return pd.read_parquet(path)


def load_all_parallel(data_dir: Path, n_batches: int = 5) -> pd.DataFrame:
    paths = sorted(data_dir.glob("*.parquet"))[:n_batches]
    with ThreadPoolExecutor(max_workers=None) as executor:
        dfs = list(executor.map(load_batch, [str(p) for p in paths]))
    return pd.concat(dfs, ignore_index=True)


def build_pipeline(n_jobs: int = -1) -> Pipeline:
    return Pipeline(
        [
            (
                "column_transformer",
                ColumnTransformer(
                    [
                        ("ohe", OneHotEncoder(handle_unknown="ignore"), ["key", "mode", "genre"]),
                        ("numerical", "passthrough", PARAM_COLS),
                    ]
                ),
            ),
            ("random_forest", RandomForestRegressor(n_jobs=n_jobs, random_state=42)),
        ]
    )


# ── Funciones de las tareas de Airflow ───────────────────────────────────────


def task_load_data_fn(**context):
    """Carga 5 batches en paralelo, los guarda en disco y pasa la ruta por XCom."""
    df = load_all_parallel(DATA_DIR, n_batches=5)
    df.to_parquet(OUTPUT_PATH)
    context["ti"].xcom_push(key="data_path", value=str(OUTPUT_PATH))
    print(f"Datos cargados: {df.shape[0]} filas guardadas en {OUTPUT_PATH}")


def task_train_model_fn(**context):
    """Recupera la ruta desde XCom, carga los datos y entrena el pipeline."""
    data_path = context["ti"].xcom_pull(key="data_path", task_ids="load_data")
    df = pd.read_parquet(data_path)

    X = df[PARAM_COLS + ["key", "mode", "genre"]]
    y = df["valence"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    pipeline = build_pipeline(n_jobs=-1)
    t0 = time.perf_counter()
    pipeline.fit(X_train, y_train)
    print(f"Modelo entrenado en {time.perf_counter() - t0:.1f}s sobre {X_train.shape[0]} filas")


# ── Definición del DAG ────────────────────────────────────────────────────────

with DAG(
    dag_id="spotify_pipeline",
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
    tags=["mds7202", "spotify"],
) as dag:
    load_data = PythonOperator(
        task_id="load_data",
        python_callable=task_load_data_fn,
    )

    train_model = PythonOperator(
        task_id="train_model",
        python_callable=task_train_model_fn,
    )

    load_data >> train_model


Overwriting dags/spotify_pipeline_dag.py


Una vez guardado el archivo, ejecuta el DAG con:

```bash
airflow standalone
```


### Pega aquí el output de las steps del DAG

El DAG se ejecutó con `airflow dags test spotify_pipeline 2026-01-01`. Las dos tareas corrieron en orden (load_data → train_model) y ambas terminaron en estado `success`.

- Step 1 (load_data):

```
[DAG TEST] starting task_id=load_data map_index=-1
Current task name:load_data
Dag name:spotify_pipeline
Datos cargados: 50000 filas guardadas en /tmp/spotify_data.parquet
Task instance in success state
[DAG TEST] end task task_id=load_data map_index=-1
```

- Step 2 (train_model):

```
[DAG TEST] starting task_id=train_model map_index=-1
Current task name:train_model
Dag name:spotify_pipeline
Modelo entrenado en 6.7s sobre 40000 filas
Task instance in success state
[DAG TEST] end task task_id=train_model map_index=-1

Marking run <DagRun spotify_pipeline @ 2026-01-01 00:00:00+00:00 ...> successful
DagRun Finished: dag_id=spotify_pipeline, logical_date=2026-01-01 00:00:00+00:00, state=success, run_type=manual
```


### Preguntas [1.2 Puntos]

1. ¿Qué es un **DAG**? ¿Qué significa que sea *Directed* (dirigido) y *Acyclic* (acíclico)? ¿Por qué importa la propiedad acíclica en un pipeline de datos?
2. ¿Qué es **Apache Airflow**? ¿Para qué tipo de problemas está diseñado y cuál es su unidad mínima de trabajo?
3. ¿Qué son los **Operators**? ¿Qué diferencia hay entre `PythonOperator` y `BashOperator`? ¿Cuándo usarías cada uno?
4. ¿Qué es **XCom** en Airflow? ¿Cómo funciona internamente (¿dónde se almacena?)? ¿Por qué **no** es adecuado para pasar DataFrames grandes entre tareas?
5. ¿Qué alternativa concreta usaste para pasar el DataFrame entre `load_data` y `train_model`? ¿Cuál sería la alternativa recomendada en producción (S3, GCS, DVC…)?
6. ¿Qué es el parámetro `schedule` de un DAG? ¿Cómo lo configurarías para que corra todos los días a las 3 AM?
7. ¿Qué diferencia hay entre Airflow y otras herramientas como **Prefect**, **Dagster**, **Luigi**, **Kubeflow**? ¿Cuál es la principal crítica que se le hace a Airflow?
8. ¿Por qué conviene orquestar el pipeline en Airflow en vez de simplemente ejecutar un script Python end-to-end?
9. ¿Qué pasa si `load_data` falla a mitad de camino? ¿Airflow reintenta automáticamente? ¿Cómo controlarías el número máximo de reintentos?
10. ¿Qué ventaja tiene que las tareas estén separadas (carga y entrenamiento) vs. una sola tarea monolítica, desde el punto de vista de debugging y eficiencia?
11. ¿Cómo podemos alertar si es que algún paso falla? ¿O si la pipeline se ejecuta correctamente?
12. En un pipeline de producción real, ¿qué otras tareas añadirías al DAG?

1. Un DAG (Directed Acyclic Graph) es un grafo dirigido y acíclico que representa el pipeline: cada nodo es una tarea y cada arista una dependencia. Dirigido significa que las aristas tienen un sentido (A va antes que B, no al revés), y acíclico que no hay ciclos, o sea que siguiendo las flechas no se puede volver a una tarea ya ejecutada. La propiedad acíclica importa porque garantiza que el pipeline tenga un orden de ejecución bien definido y que termine: si hubiera un ciclo, dos tareas dependerían mutuamente y ninguna podría arrancar.

2. Apache Airflow es una plataforma de orquestación de workflows que permite definir, programar y monitorear pipelines de datos y de ML como código. Está pensado para automatizar flujos con dependencias entre pasos, ejecución periódica, reintentos y monitoreo. Su unidad mínima de trabajo es la tarea (task), que es una instancia de un operator dentro de un DAG.

3. Los Operators son plantillas que definen un tipo de trabajo a ejecutar; al instanciarlos dentro de un DAG se convierten en tareas. PythonOperator ejecuta una función de Python y BashOperator ejecuta un comando de shell. Usaría PythonOperator cuando la lógica está en Python (cargar datos, entrenar un modelo) y BashOperator para invocar scripts o herramientas de línea de comandos (mover archivos, correr un ejecutable, lanzar un comando del sistema).

4. XCom (cross-communication) es el mecanismo de Airflow para pasar datos pequeños entre tareas. Internamente se guarda en la base de datos de metadata de Airflow, como una fila con el valor serializado. Por eso no sirve para DataFrames grandes: cargaría la base de metadata con objetos pesados, la serialización y deserialización serían costosas y hay límites de tamaño. XCom está pensado para cosas chicas como rutas, ids o flags, no para datasets completos.

5. Para pasar el DataFrame guardé los datos en disco como Parquet (en OUTPUT_PATH) y pasé por XCom solo la ruta del archivo, que es un string pequeño; así la tarea de entrenamiento lee el DataFrame desde esa ruta. En producción lo recomendado sería un almacenamiento externo compartido, como S3 o GCS (o un sistema de versionado de datos como DVC), de modo que las tareas accedan al mismo archivo de forma confiable aunque corran en workers distintos.

6. schedule define cada cuánto se ejecuta el DAG, normalmente con una expresión cron (o None para ejecución manual). Para que corra todos los días a las 3 AM usaría schedule="0 3 * * *".

7. Airflow, Prefect, Dagster, Luigi y Kubeflow son orquestadores con enfoques distintos. Luigi es más antiguo y simple, orientado a dependencias entre tareas. Prefect y Dagster son más modernos: Prefect apunta a una API más pythónica y dinámica, y Dagster pone el foco en los data assets y en la testeabilidad. Kubeflow está centrado en pipelines de ML sobre Kubernetes. La crítica más común a Airflow es que sus DAGs son relativamente estáticos y pesados de configurar, que el scheduler puede ser complejo de operar y que históricamente cuesta pasar datos entre tareas y armar flujos dinámicos.

8. Porque un script end-to-end es todo o nada: si falla a la mitad hay que volver a correrlo entero, sin reintentos ni visibilidad de en qué paso se cayó. Airflow aporta dependencias explícitas entre tareas, ejecución programada, reintentos automáticos, logs y monitoreo por tarea, y la posibilidad de reanudar desde donde falló. Eso lo vuelve mucho más robusto y mantenible para producción.

9. Si load_data falla, la tarea queda marcada como fallida y train_model no se ejecuta, porque depende de ella. Airflow reintenta automáticamente solo si se configura: con el parámetro retries (número máximo de reintentos) y retry_delay (espera entre intentos), que se pasan a la tarea o como default_args del DAG. Si no se configuran, no reintenta.

10. Separar las tareas facilita el debugging porque cada una tiene sus propios logs y estado, así que si algo falla se ve exactamente en qué paso fue y se puede reintentar solo esa tarea sin repetir las demás. En eficiencia, permite reusar el resultado de un paso ya completado (no recargar los datos si solo falló el entrenamiento) y, en pipelines más grandes, paralelizar tareas independientes. Una tarea monolítica no da nada de eso: si falla al final, hay que rehacer todo.

11. Airflow permite configurar alertas ante fallos: el parámetro email_on_failure junto con email envía un correo cuando una tarea falla, y también se pueden usar callbacks (on_failure_callback y on_success_callback) para notificar por otros medios, por ejemplo mandando un mensaje a Slack o a un webhook cuando la tarea falla o cuando el pipeline termina bien.

12. En un pipeline real agregaría tareas de validación de datos (chequear esquema, nulos y rangos) antes de entrenar, ingeniería de features, evaluación del modelo con métricas y comparación contra el modelo en producción, registro del modelo y sus métricas (por ejemplo en MLflow), despliegue del modelo si supera un umbral, y notificaciones al final. También tareas de limpieza de archivos temporales y, eventualmente, monitoreo de data drift.


# Conclusión

Eso ha sido todo para el lab de hoy. Recuerda que el laboratorio tiene un plazo de entrega de una semana. Cualquier duda, no dudes en contactarnos por el foro de U-Cursos.

<p align="center">
  <img src="https://media.giphy.com/media/l0HlBO7eyXzSZkJri/giphy.gif" width="300">
</p>